# Deep Agents (Orchastrator + SubAgents)

In [3]:
# Imports
import os, subprocess, tempfile
from typing import Literal
from tavily import TavilyClient # Basic Real-Time Search for AI Agents
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"..\config\.env")

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

GOOGLE_MODEL = os.getenv("GOOGLE_MODEL")
GROQ_MODEL = os.getenv("GROQ_MODEL")

## Tools

In [ ]:
# ---------- Tools ----------
def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Search the web for current information."""
    return tavily_client.search(
        query, max_results=max_results,
        include_raw_content=include_raw_content, topic=topic,
    )

def run_python_code(code: str) -> str:
    """Execute a Python snippet in an isolated subprocess and return stdout/stderr."""
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(code)
        path = f.name
    try:
        result = subprocess.run(["python", path], capture_output=True, text=True, timeout=30)
        out = result.stdout
        if result.stderr:
            out += f"\n--- STDERR ---\n{result.stderr}"
        return out or "(no output)"
    finally:
        os.remove(path)

## Models

In [ ]:
# ---------- Models (swap freely per agent) ----------

orchestrator_model = init_chat_model(GROQ_MODEL)
research_model     = init_chat_model(GROQ_MODEL)
coding_model       = init_chat_model(GROQ_MODEL)
writing_model      = init_chat_model(GROQ_MODEL)